# core

> ledger.csv backend — pure logic and atomic CSV I/O (see `DESIGN.md` §3). The web layer lives in `01_app.ipynb`.

In [ ]:
#| default_exp core

In [ ]:
#| export
import os
from datetime import date, timedelta
from pathlib import Path

import pandas as pd

In [ ]:
#| hide
import tempfile

## `save_full`

In [ ]:
#| export
def save_full(
    df:pd.DataFrame,  # ledger frame — index=date, columns=item names, values=bool
    path:Path):  # live CSV; a .tmp sibling in the same dir is used for the swap
    "Atomic write: a mid-write crash must never corrupt the live file."
    path.parent.mkdir(exist_ok=True)
    tmp = path.with_suffix(".tmp")
    df.to_csv(tmp)
    os.replace(tmp, path)

In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "ledger.csv"
    df = pd.DataFrame({"読書": [True, False]},
                      index=pd.Index([date(2026, 8, 23), date(2026, 8, 24)], name="date"))
    save_full(df, p)
    assert p.exists() and not p.with_suffix(".tmp").exists()
    back = pd.read_csv(p, index_col=0)
    assert list(back.columns) == ["読書"]
    assert back["読書"].tolist() == [True, False]

## `load_full`

In [ ]:
#| export
def load_full(
    path:Path,  # ledger CSV location
    default_items:list,  # columns for first-run auto-create
    today:date,  # index row for first-run auto-create
) -> pd.DataFrame:
    "Full history; blank cells read as False; auto-creates the CSV on first run."
    if not path.exists():
        df = pd.DataFrame(False, index=pd.Index([today], name="date"),
                          columns=default_items)
        save_full(df, path)
        return df
    df = pd.read_csv(path, index_col=0)
    df = df.fillna(False).astype(bool)
    df.index = pd.Index([date.fromisoformat(d) for d in df.index], name="date")
    return df

In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "ledger.csv"
    t = date(2026, 8, 24)

    # first run: auto-create, everything False
    df = load_full(p, ["読書", "運動"], t)
    assert p.exists()
    assert list(df.columns) == ["読書", "運動"]
    assert list(df.index) == [t]
    assert df.dtypes.eq(bool).all() and not df.values.any()

    # blank cells (e.g. a hand-added column) read as False, never True
    p.write_text("date,読書,運動\n2026-08-23,True,\n2026-08-24,,False\n")
    df = load_full(p, [], t)
    assert df.dtypes.eq(bool).all()
    assert df.at[date(2026, 8, 23), "読書"] == True
    assert df.at[date(2026, 8, 23), "運動"] == False
    assert df.at[date(2026, 8, 24), "読書"] == False
    assert df.at[date(2026, 8, 24), "運動"] == False

## `window`

In [ ]:
#| export
def window(
    full:pd.DataFrame,  # full history from `load_full`
    today:date,  # last day of the window
    days:int=10,  # window length
) -> pd.DataFrame:
    "Trailing `days`-day view ending today; dates missing from history are False."
    idx = [today - timedelta(days=i) for i in range(days - 1, -1, -1)]
    return full.reindex(pd.Index(idx, name="date"), fill_value=False)

In [ ]:
t = date(2026, 8, 24)
full = pd.DataFrame({"読書": [True]},
                    index=pd.Index([date(2026, 8, 20)], name="date")).astype(bool)
w = window(full, t, 10)
assert len(w) == 10
assert list(w.index)[0] == date(2026, 8, 15) and list(w.index)[-1] == t
assert w.at[date(2026, 8, 20), "読書"] == True
assert w.at[t, "読書"] == False  # date missing from history -> False
assert w.dtypes.eq(bool).all()

## `current_streak`

In [ ]:
#| export
def current_streak(
    full:pd.DataFrame,  # full history from `load_full`
    item:str,  # column name
    today:date,
) -> int:
    "Consecutive checked days ending today — or yesterday, if today is still unchecked."
    col = full[item]
    d = today
    if not col.get(d, False):  # today unchecked -> streak starts from yesterday
        d -= timedelta(days=1)
    streak = 0
    while col.get(d, False):
        streak += 1
        d -= timedelta(days=1)
    return streak

In [ ]:
t = date(2026, 8, 24)
idx = [date(2026, 8, n) for n in range(10, 25)]
full = pd.DataFrame({"x": [d.day >= 12 for d in idx]}, index=pd.Index(idx, name="date"))
assert current_streak(full, "x", t) == 13  # regression: not capped at the 10-day window

# today unchecked -> streak starts from yesterday
full.at[t, "x"] = False
assert current_streak(full, "x", t) == 12

# a gap breaks the streak
full.at[date(2026, 8, 20), "x"] = False
assert current_streak(full, "x", t) == 3

# empty history
empty = pd.DataFrame({"x": []}, index=pd.Index([], name="date")).astype(bool)
assert current_streak(empty, "x", t) == 0

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()